# Урок 6.1. Когда дообучать модель: LoRA на пальцах

Модуль 6 «Дообучение через LoRA» курса «LLM для Python-разработчиков».

Пять модулей мы учили модель отвечать **правильно**: давали ей знания через RAG,
защищали от инъекций, запускали локально. Но есть класс задач, где проблема не
в знаниях, а в **форме**: модель должна отвечать строго определённым голосом,
в строго определённой структуре — каждый раз, без напоминаний. Здесь промпт
начинает проигрывать, и на сцену выходит дообучение (fine-tuning).

🎯 **Цели урока:**

- Понять, какие задачи решает дообучение, а какие — RAG и промптинг (матрица выбора).
- Убедиться живым экспериментом, что жёсткий стиль промптом держится плохо.
- Разобрать LoRA (Low-Rank Adaptation) «на пальцах»: что за матрицы A и B, что такое ранг r и alpha.
- Посчитать на реальных числах qwen2.5:3b, почему полное дообучение нам недоступно, а LoRA — вполне.
- Узнать, что такое QLoRA и почему бесплатного Colab хватает для обучения 3B-модели.

**Что понадобится:** Ollama с qwen2.5:3b (модуль 1/5). GPU не нужен — все расчёты
урока делаются на калькуляторе, а обучать мы будем в уроке 6.2 на бесплатном Colab.

## Где мы в курсе

| Модуль | Что умеем |
|---|---|
| 1. Основы LLM | промпты, few-shot, параметры, свой CLI |
| 2. Эмбеддинги и векторные БД | поиск по смыслу, Qdrant, свой `kb-search` |
| 3. RAG | ретривер + генератор, eval, guardrails, агенты |
| 4. Продвинутый RAG | безопасность, мониторинг, reranking, права доступа |
| 5. Локальные LLM | Ollama в проде, llama.cpp, GGUF, свой прокси |
| **6. Дообучение (вы здесь)** | **вшиваем поведение прямо в веса модели** |

План модуля: в этом уроке решаем, **когда** дообучать и как устроена LoRA;
в 6.2 готовим датасет и обучаем адаптер на Colab; в 6.3 возвращаем результат
домой — merge → GGUF → Modelfile → Ollama; в 6.4 закрепляем.

Пятый модуль закончился словами «модель — это артефакт с digest». В этом модуле
мы впервые будем этот артефакт **менять**.

## 1. Задача: фирменный голос поддержки

К нам пришёл новый заказчик — хостинг-провайдер «Пингвин.Хост» 🐧. У них есть
регламент поддержки: каждый ответ клиенту обязан выглядеть так:

```
Здравствуйте! Поддержка «Пингвин.Хост» на связи. 🐧

Что происходит: <короткий диагноз своими словами>

Что делать:
1. <шаг в терминах их панели управления>
2. ...

Если шаги не помогут — просто ответьте на это сообщение, тикет остаётся открытым.
```

Заказчик хочет бота, который отвечает **только так**. Обратите внимание, чем это
отличается от «Векторики» из модулей 2–4: там модель не знала **фактов** (сколько
дней отпуска — не угадаешь), и мы приносили ей знания через RAG. Здесь факты
общеизвестные (что такое 502, как работает SSL) — не хватает **формы**: приветствия,
структуры, финальной строчки, пингвина.

У заказчика есть 45 эталонных ответов из тикет-системы — `data/train.jsonl`.
Посмотрим на них.

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("data")
train = [json.loads(line) for line in
         (DATA_DIR / "train.jsonl").read_text(encoding="utf-8").splitlines()]

print(f"Примеров: {len(train)}, ключи: {sorted(train[0])}\n")
print("--- Пример №1 " + "-" * 50)
print("ВОПРОС:", train[0]["instruction"])
print("ОТВЕТ:")
print(train[0]["output"])

# пять якорей фирменного стиля - по ним будем мерить ВСЁ в этом модуле
STYLE_CHECKS = {
    "приветствие":    lambda t: "Поддержка «Пингвин.Хост» на связи" in t,
    "пингвин \U0001F427":  lambda t: "\U0001F427" in t,
    "что происходит": lambda t: "Что происходит:" in t,
    "что делать":     lambda t: "Что делать:" in t,
    "финалка":        lambda t: "тикет остаётся открытым" in t,
}

def style_score(text: str) -> int:
    """Сколько якорей фирменного стиля есть в ответе (0-5)."""
    return sum(check(text) for check in STYLE_CHECKS.values())

anchors_ok = sum(style_score(ex["output"]) == 5 for ex in train)
lengths = sorted(len(ex["output"]) for ex in train)
print("-" * 64)
print(f"Все 5 якорей стиля: у {anchors_ok} из {len(train)} примеров")
print(f"Длина ответа: min {lengths[0]}, медиана {lengths[len(lengths)//2]}, max {lengths[-1]} симв.")

45 примеров, у всех — идеальные 5/5 якорей и близкие длины. Это **датасет про
форму**: вопросы разные (от 502-й до возврата денег), а скелет ответа одинаковый.

Функция `style_score` — это наш **eval** (привычка урока 1.7 никуда не делась):
прежде чем что-то улучшать, договариваемся, чем меряем. Пять якорей — грубая
метрика, но честная: их нельзя «почти выполнить».

Теперь главный вопрос урока: как заставить модель отвечать в этом стиле?
По лестнице из урока 1.7 — сначала пробуем самое дешёвое: промпт.

In [ ]:
import httpx

OLLAMA_URL = "http://127.0.0.1:11434"   # 127.0.0.1, не localhost - урок 3.2
MODEL = "qwen2.5:3b"

def chat(messages: list[dict], num_predict: int = 350):
    """Один вызов /api/chat; возвращает (ответ, токенов_промпта, токенов_ответа)."""
    response = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL, "messages": messages, "stream": False,
        "options": {"temperature": 0, "num_predict": num_predict},
    }, timeout=180)
    response.raise_for_status()
    data = response.json()
    return (data["message"]["content"],
            data.get("prompt_eval_count", 0), data.get("eval_count", 0))

# отложенные вопросы: их НЕТ в train.jsonl - на них меряем честно
test_questions = [json.loads(line)["question"] for line in
                  (DATA_DIR / "test_questions.jsonl").read_text(encoding="utf-8").splitlines()]

SYSTEM_SHORT = "Ты — сотрудник поддержки хостинга «Пингвин.Хост». Отвечай на вопросы клиентов."

try:
    question = test_questions[3]  # про несвоевременную оплату
    print("ВОПРОС:", question, "\n")
    answer, p_tokens, e_tokens = chat([
        {"role": "system", "content": SYSTEM_SHORT},
        {"role": "user", "content": question},
    ])
    print(f"=== ZERO-SHOT (промпт {p_tokens} ток.) ===")
    print(answer)
    print(f"\nСтиль: {style_score(answer)}/5")
except httpx.HTTPError as exc:
    print(f"[пропущено: Ollama недоступна - {exc.__class__.__name__}. "
          f"Запустите её и выполните ollama pull {MODEL}]")

В моём прогоне — **0/5 якорей**. Хуже того: модель уверенно перечисляет
«варианты», которых мы ей не давали, — «перенос на следующий тарифный план»,
«временный доступ без оплаты». Это не сотрудник «Пингвин.Хост» — это импровизация
на тему хостинга: ни одного якоря формата и выдуманные политики компании.

Ответы qwen на GPU не бит-в-бит воспроизводимы даже при `temperature: 0`
(мы выяснили это ещё в уроке 3.3) — у вас текст может отличаться. Но 5/5 якорей
zero-shot не наберёт: модель не телепат, она в глаза не видела наш регламент.

Ок, лестница урока 1.7: следующая ступень — **показать примеры**. Few-shot.

In [ ]:
# few-shot: три эталонных ответа прямо в диалог (приём урока 1.7)
few_shot = []
for ex in train[:3]:
    few_shot.append({"role": "user", "content": ex["instruction"]})
    few_shot.append({"role": "assistant", "content": ex["output"]})

try:
    for question in (test_questions[3], test_questions[0]):
        answer, p_tokens, _ = chat([
            {"role": "system", "content": SYSTEM_SHORT},
            *few_shot,
            {"role": "user", "content": question},
        ])
        print(f"ВОПРОС: {question}")
        print(f"  промпт: {p_tokens} ток.  стиль: {style_score(answer)}/5")
        print("  " + answer[:220].replace("\n", " ") + "...\n")
except httpx.HTTPError as exc:
    print(f"[пропущено: Ollama недоступна - {exc.__class__.__name__}]")

Мой прогон (у вас цифры могут отличаться — но картина типична):

| | промпт | стиль |
|---|---|---|
| zero-shot | 66 ток. | 0/5 |
| few-shot ×3, вопрос про оплату | 661 ток. | 0/5 |
| few-shot ×3, вопрос про таймаут | 658 ток. | 2/5 |

Три проблемы, каждая — принципиальная:

1. **Стиль плавает.** На одном вопросе модель скопировала приветствие и пингвина,
   на другом — проигнорировала все пять якорей. Few-shot *подсказывает* форму,
   но не *обязывает*: у 3B-модели своё представление о хорошем ответе, и оно
   перевешивает.
2. **Промпт вырос в 10 раз** — с 66 до ~660 токенов. Эти токены мы платим
   (временем prefill — урок 5.2) **на каждом запросе**, вечно.
3. **Выдумки никуда не делись.** В ответе про оплату модель изобрела «режим
   ожидания (Hold)», которого нет ни в примерах, ни в вопросе (в другом прогоне
   на этом же месте была «система аварийного доступа» — выдумка стабильна,
   меняется только её содержание). Примеры в промпте она использует не только
   как образец формы, но и как трамплин для «фактов».

Можно бороться: примеров не 3, а 10, добавить жёстких инструкций CAPSLOCK'ом...
Промпт распухнет ещё сильнее, а гарантий так и не появится. Пора признать:
мы пытаемся **промптом** решить задачу, которая решается **весами**.

## 2. Матрица выбора: промпт, RAG или дообучение

Как матрицы из уроков 1.3 и 5.1, только теперь по вертикали — способы «научить» модель:

| Способ | Что меняем | Хорош для | Плох для | Цена запроса | Цена входа |
|---|---|---|---|---|---|
| Инструкция в промпте | текст запроса | разовые задачи, прототипы | жёсткие форматы, большие правила | +десятки токенов | ~0 |
| Few-shot (1.7) | текст запроса | формат ответов «по мотивам» | стабильность, много правил | +сотни токенов | ~0 |
| RAG (модули 2–4) | контекст запроса | **знания**: факты, документы, всё что устаревает | стиль, форма, тон | +сотни токенов | дни (пайплайн) |
| Fine-tuning (этот модуль) | **веса модели** | **форма**: стиль, структура, тон, жаргон домена | свежие факты | +0 токенов | часы (датасет + обучение) |

Главное правило выбора, выстраданное индустрией:

> **Знания — в RAG, поведение — в веса.**

Дообучать модель ради фактов — антипаттерн: факты устаревают (переобучать?),
их много (весов не напасёшься), и выученное невозможно процитировать с источником
(прощай, grounding из урока 3.1). А вот форму RAG'ом не решить: сколько ни клади
документов в контекст, «голос» модели останется её собственным.

Частый продакшен-паттерн — **гибрид**: LoRA учит модель фирменному голосу и формату,
а RAG приносит в контекст свежие факты. Стиль из весов + знания из поиска.

И проверка на здравый смысл перед стартом любого дообучения: **а есть ли у вас
45+ эталонных примеров?** Нет примеров — нечему учить; сначала данные, потом веса.

## 3. Почему не полное дообучение: арифметика

«Дообучить модель» звучит просто: продолжить градиентный спуск на своих данных.
Но посчитаем, что это значит для нашей qwen2.5:3b — все числа возьмём живьём
из `/api/show` (тот же приём «спроси процесс, а не документацию», что и в уроке 5.3).

In [ ]:
try:
    info = httpx.post(f"{OLLAMA_URL}/api/show",
                      json={"model": MODEL}, timeout=10).json()["model_info"]
    arch = {
        "params_total": info["general.parameter_count"],
        "hidden":       info["qwen2.embedding_length"],
        "blocks":       info["qwen2.block_count"],
        "heads_q":      info["qwen2.attention.head_count"],
        "heads_kv":     info["qwen2.attention.head_count_kv"],
        "ffn":          info["qwen2.feed_forward_length"],
    }
except httpx.HTTPError:
    print("[Ollama недоступна - использую сохранённые числа qwen2.5:3b]")
    arch = {"params_total": 3_085_938_688, "hidden": 2048, "blocks": 36,
            "heads_q": 16, "heads_kv": 2, "ffn": 11008}

P = arch["params_total"]
print(f"qwen2.5:3b: {P:,} параметров, {arch['blocks']} блоков, "
      f"hidden {arch['hidden']}, GQA {arch['heads_q']}/{arch['heads_kv']} голов\n")

# классическая смета памяти mixed-precision обучения с Adam, байт на параметр:
BYTES = {
    "веса fp16":            2,
    "градиенты fp16":       2,
    "Adam m (fp32)":        4,
    "Adam v (fp32)":        4,
    "мастер-веса fp32":     4,
}
print("Полное дообучение (fp16 + Adam), без активаций:")
total = 0
for name, b in BYTES.items():
    gb = P * b / 2**30
    total += gb
    print(f"  {name:<22} {b} Б/парам  = {gb:6.1f} ГБ")
print(f"  {'ИТОГО':<22}           = {total:6.1f} ГБ + активации")
print(f"\nНаша GPU: 8 ГБ. Colab T4: 16 ГБ. Не влезает никуда.")

**~46 ГБ** только на веса и оптимизатор — не считая активаций, которые при
длинных примерах добавляют ещё гигабайты. Полное дообучение 3B-модели — это
аренда A100 на 80 ГБ. А ведь мы хотели «слегка поправить стиль».

Несоответствие масштаба задачи и цены — то самое место, где появляется LoRA.
Интуиция такая: чтобы научить модель здороваться пингвином, не нужно двигать
все 3 миллиарда параметров. Нужна **маленькая поправка** к большим матрицам.

## 4. LoRA на пальцах

Внутри трансформера почти все параметры — это большие матрицы весов `W`
(проекции внимания q/k/v/o и слои FFN). Полное дообучение меняет каждую `W`
целиком. LoRA (Low-Rank Adaptation) вместо этого **замораживает** `W`
и учит рядом маленькую поправку:

```
                    было:  y = W·x            (W - огромная, заморожена ❄)

                   стало:  y = W·x + (alpha/r)·B·A·x
                                              │   │
        A: маленькая (r × in) ────────────────┘   │
        B: маленькая (out × r) ───────────────────┘
```

- `W` имеет размер, скажем, 2048×2048 = 4.2 млн параметров.
- `A` (r×2048) и `B` (2048×r) при ранге r=16 — всего 16·(2048+2048) = 65 тысяч.
  **В 64 раза меньше**, а поправка действует на всю матрицу.
- `r` — **ранг** поправки: насколько «богатой» она может быть. Стиль и формат —
  простые правки, хватает r=8–16. Новый язык или домен — r=64+.
- `alpha` — масштаб: итоговая поправка умножается на `alpha/r`. Стандарт —
  `alpha = r` или `alpha = 2r` (масштаб 1–2).
- Инициализация хитрая: `A` — случайная, `B` — **нули**. Поэтому в нулевой момент
  `B·A = 0`, и обучение стартует ровно с базовой модели, ничего не сломав.

Почему низкого ранга хватает? Эмпирический факт из статьи LoRA: изменения весов
при дообучении на узкую задачу имеют **низкий эффективный ранг** — вся «разница»
между базовой моделью и дообученной хорошо сжимается в произведение двух узких
матриц. Мы не учим модель заново — мы легонько поворачиваем то, что уже есть.

`target_modules` — на какие матрицы вешать адаптеры. Минимум — проекции внимания
(`q_proj`, `k_proj`, `v_proj`, `o_proj`); современный дефолт (и дефолт unsloth) —
все линейные слои, включая FFN (`gate_proj`, `up_proj`, `down_proj`). Посчитаем
оба варианта на реальных размерах qwen2.5:3b.

In [ ]:
hidden, blocks, ffn = arch["hidden"], arch["blocks"], arch["ffn"]
head_dim = 128                                  # у qwen2.5 фиксирован
kv_dim = arch["heads_kv"] * head_dim            # GQA: k/v уже, чем q (урок 5.2)

# (вход, выход) каждой линейной матрицы одного блока трансформера
LAYERS = {
    "q_proj":    (hidden, hidden),
    "k_proj":    (hidden, kv_dim),
    "v_proj":    (hidden, kv_dim),
    "o_proj":    (hidden, hidden),
    "gate_proj": (hidden, ffn),
    "up_proj":   (hidden, ffn),
    "down_proj": (ffn, hidden),
}
ATTN_ONLY = ("q_proj", "k_proj", "v_proj", "o_proj")

def lora_params(r: int, targets) -> int:
    """A: r x in, B: out x r  =>  r * (in + out) на матрицу, на каждый блок."""
    return blocks * sum(r * (i + o) for name, (i, o) in LAYERS.items()
                        if name in targets)

print(f"{'r':>4} | {'attention (q,k,v,o)':>22} | {'все linear (+FFN)':>20}")
for r in (8, 16, 32, 64):
    attn = lora_params(r, ATTN_ONLY)
    full = lora_params(r, LAYERS)
    print(f"{r:>4} | {attn:>12,} ({attn/P:6.2%}) | {full:>10,} ({full/P:6.2%})")

r = 16
adapter_mb = lora_params(r, LAYERS) * 2 / 2**20   # fp16 = 2 байта
print(f"\nАдаптер r=16 на все linear: ~{adapter_mb:.0f} МБ в fp16")
print(f"Для сравнения, сама модель Q4_K_M: ~1900 МБ")

LoRA r=16 на все линейные слои — **~30 млн обучаемых параметров, меньше 1%**
модели, а сам адаптер весит ~57 МБ. Его можно прислать по почте, хранить
в git LFS, накладывать и снимать с базовой модели как плагин.

Обратите внимание на строку `k_proj`/`v_proj`: благодаря GQA (урок 5.2 —
2 KV-головы против 16 Q-голов) эти матрицы узкие, и адаптеры на них почти
бесплатны.

Правило выбора `r` для старта: **начинайте с 16**. Меньше — если данных мало
и задача простая (наш случай: 45 примеров чистой формы — r=16 с запасом);
больше — только когда eval покажет, что модель «не тянет» задачу. Увеличение
ранга на маленьком датасете добавляет не качество, а переобучение.

## 5. QLoRA: тот же адаптер, база в 4 битах

LoRA сняла проблему градиентов и оптимизатора (они нужны только для 1% весов),
но замороженная база всё ещё лежит в памяти в fp16 — 6.2 ГБ для 3B. На пределе
даже для нашей 8-гиговой карты, если вспомнить про активации.

**QLoRA** делает последний шаг: замороженную базу квантует в **4 бита** (формат
NF4 — «нормальный float», оптимизированный под колокол распределения весов,
плюс double quantization — квантуются даже константы квантования). Адаптеры
и градиенты остаются в bf16/fp16: учится по-прежнему «полноценная» маленькая
поправка, просто поверх сжатой базы. Квантизация из урока 5.2 — теперь не только
для инференса, но и для обучения.

Смета памяти для qwen2.5:3b (грубо, без активаций):

| Схема | База | Градиенты+Adam | Итого | Куда влезает |
|---|---|---|---|---|
| Полный FT fp16 | 6.2 ГБ | ~40 ГБ | **~46 ГБ** | A100 80 ГБ |
| LoRA r=16 | 6.2 ГБ (fp16 ❄) | ~0.4 ГБ | **~7–9 ГБ** | T4 16 ГБ, впритык 8 ГБ |
| QLoRA r=16 | ~1.9 ГБ (4 бита ❄) | ~0.4 ГБ | **~3–5 ГБ** | свободно T4 и наши 8 ГБ |

Цена — чуть медленнее обучение (деквантизация на лету) и микроскопическая потеря
точности базы. Для задачи «выучи наш формат» это несущественно.

Именно поэтому весь урок 6.2 пройдёт на **бесплатном Colab** (T4, 16 ГБ): QLoRA
3B-модели там помещается с большим запасом. Владельцы GPU от 8 ГБ смогут повторить
то же самое локально — но это опциональный путь, не обязательный.

## 6. Три страха дообучения

**Катастрофическое забывание (catastrophic forgetting).** При полном дообучении
на узких данных модель может «забыть» общие навыки — вы учили её пингвинам,
а она разучилась писать код. LoRA защищена конструктивно: базовые веса заморожены,
адаптер меняет поведение поверх них — и его всегда можно снять. Но перегнуть
можно и с LoRA: слишком большой `r`, слишком много эпох — и модель начнёт
совать «Что делать:» в ответы про столицу Франции. Проверка общих навыков
после обучения — обязательный пункт приёмки (сделаем в 6.3).

**Переобучение (overfitting).** 45 примеров — это мало. Модель может не выучить
*правило* «начинай с приветствия», а *зазубрить* сами ответы дословно. Симптом:
на вопросы из train отвечает идеально, на новые — цитирует куски чужих ответов
про SSL в вопросе про почту. Лечение: отложенные вопросы (наш
`test_questions.jsonl` — их модель не увидит при обучении), 1–3 эпохи, малый `r`.

**Дрейф базы.** Адаптер обучен поверх **конкретных** весов. Выйдет qwen2.6 —
и ваш адаптер к ней не приложится (а если приложится по размерам — поведение
непредсказуемо). Адаптер + digest базовой модели (урок 5.6) должны храниться
вместе, как миграция и версия схемы БД.

## ⚠️ Частые ошибки

1. **Учить LoRA'й факты.** «Дообучим модель на нашей документации, и RAG не нужен» —
   не нужен, пока документация не изменилась (а она уже изменилась). Знания — в RAG.
2. **`r` побольше «для надёжности»**. На 45 примерах r=256 — это способ выучить
   датасет наизусть, а не стиль. Начинайте с 16.
3. **Нет отложенного набора.** Мерить стиль на тех же примерах, на которых учили, —
   это eval на train (смертный грех из урока 1.7 всё ещё грех).
4. **Сравнивать «до» и «после» разными промптами.** После обучения хочется убрать
   few-shot из промпта — уберите его и из замера «до», иначе сравниваете яблоки
   с пингвинами.
5. **Забыть, что адаптер привязан к базе.** «Скачал qwen2.5:3b заново, стало странно» —
   а тег `:3b` мог уже указывать на другой digest (урок 5.6 предупреждал).
6. **Дообучать то, что решается системным промптом.** Если стиль держится
   инструкцией в 30 токенов — не усложняйте. Мы дообучаем потому, что живой
   эксперимент выше показал: не держится.

## Упражнения

1. **Калькулятор для старшего брата.** У qwen2.5:7b параметры такие: hidden 3584,
   28 блоков, GQA 28/4 головы (head_dim 128), FFN 18944, всего ~7.6 млрд.
   Посчитайте параметры LoRA r=16 на все линейные слои и их долю от модели.
   Изменится ли вывод «меньше 1%»?

2. **Прикидка VRAM.** Влезет ли QLoRA-обучение qwen2.5:7b в бесплатный Colab T4
   (16 ГБ)? Посчитайте по смете из раздела 5: база в 4 битах ~0.6 байта/парам
   (с учётом констант квантования), адаптер+градиенты+Adam — по таблице, на
   активации заложите 2–4 ГБ.

3. **Классификатор задач.** Три задачи из практики — промпт, RAG или fine-tuning?
   а) бот должен знать текущие тарифы компании (меняются раз в месяц);
   б) бот должен отвечать в стиле техписателя компании и всегда возвращать JSON
      `{"answer": ..., "confidence": ...}`;
   в) бот должен обращаться к клиенту по имени из CRM.

In [ ]:
# Решение 1: qwen2.5:7b
h7, blocks7, ffn7, kv7 = 3584, 28, 18944, 4 * 128
P7 = 7_615_616_512   # ~7.6 млрд

layers7 = {
    "q_proj": (h7, h7), "k_proj": (h7, kv7), "v_proj": (h7, kv7), "o_proj": (h7, h7),
    "gate_proj": (h7, ffn7), "up_proj": (h7, ffn7), "down_proj": (ffn7, h7),
}
r = 16
lora7 = blocks7 * sum(r * (i + o) for i, o in layers7.values())
print(f"LoRA r=16 на все linear у 7B: {lora7:,} параметров = {lora7/P7:.2%} модели")
print(f"Адаптер в fp16: ~{lora7 * 2 / 2**20:.0f} МБ")
# Вывод тот же: доля даже НИЖЕ, чем у 3B (~0.5%) - у больших моделей матрицы шире,
# а LoRA растёт линейно от (in+out), тогда как сама матрица - от in*out.

**Решение 2.** База 7.6 млрд × ~0.6 Б ≈ 4.6 ГБ; адаптер ~40 млн параметров —
градиенты и Adam для него ~0.6 ГБ; активации 2–4 ГБ. Итого **~7–9 ГБ** — в T4
(16 ГБ) влезает с запасом. Вывод: бесплатного Colab хватает и для 7B через QLoRA.

**Решение 3.**
а) **RAG** (а лучше — просто подстановка таблицы тарифов в промпт): факты, которые
меняются, в веса не зашивают.
б) **Fine-tuning**: стиль + жёсткий формат — ровно наш случай (а JSON дополнительно
страхуется грамматикой из урока 5.4 — ремни и подтяжки).
в) **Ни то, ни другое**: имя клиента — это переменная в шаблоне промпта. Не всё,
что звучит как «поведение бота», требует машинного обучения.

## Мини-квиз

**1. Боту нужно отвечать на вопросы по внутренней вики (обновляется ежедневно). RAG или LoRA?**

<details><summary>Ответ</summary>

RAG. Ежедневно переобучать адаптер — дорого и бессмысленно, а главное — выученное
нельзя процитировать с источником. LoRA здесь пригодилась бы только если у ответов
должен быть особый стиль — и то в паре с RAG, а не вместо.
</details>

**2. Почему матрица B в LoRA инициализируется нулями?**

<details><summary>Ответ</summary>

Чтобы в момент старта поправка B·A равнялась нулю и модель начинала обучение
ровно с базового поведения. Случайная B сразу внесла бы шум в каждый прогон
через слой — обучение стартовало бы со «сломанной» модели.
</details>

**3. Few-shot дал 5/5 якорей стиля на трёх тестовых вопросах. Зачем всё же дообучать?**

<details><summary>Ответ</summary>

Может, и незачем — если 5/5 держится на большом отложенном наборе, а сотни лишних
токенов промпта на каждый запрос вас устраивают. Дообучение выигрывает, когда
важна стабильность формы при коротком промпте: стиль в весах не «вымывается»
длинным контекстом и не стоит токенов. Решение принимает eval, а не мода.
</details>

**4. Чем QLoRA отличается от LoRA и что теряем?**

<details><summary>Ответ</summary>

База хранится в 4 битах (NF4) вместо fp16 — память на веса падает в ~4 раза,
и 3B-модель учится даже на 8 ГБ VRAM. Адаптер по-прежнему обучается в bf16/fp16.
Теряем немного скорости (деквантизация на лету) и каплю точности базы.
</details>

**5. После дообучения модель вставляет «Что делать:» в ответ на «напиши стихотворение». Что произошло и что делать?**

<details><summary>Ответ</summary>

Перегнули: стиль «протёк» на задачи вне домена — признак переобучения (слишком
много эпох / большой r на 45 примерах). Лечение: меньше эпох, r=8–16, и приёмка
с проверкой общих навыков. Радикальный плюс LoRA: адаптер можно просто снять.
</details>

**6. Адаптер обучен на qwen2.5:3b полгода назад. Сегодня `ollama pull qwen2.5:3b` — и стиль сломался. Почему?**

<details><summary>Ответ</summary>

Тег мутабелен (урок 5.6): под `:3b` теперь другой digest — другие веса, а адаптер
математически привязан к старым. Храните digest базы рядом с адаптером и
прикалывайте базу по digest, как зависимость с точной версией.
</details>

## 🎓 Итоги

- **Знания — в RAG, поведение — в веса.** Жёсткий фирменный стиль — задача для
  дообучения: живой эксперимент показал, что few-shot держит его нестабильно
  (0–2 якоря из 5) и стоит ~660 токенов на каждый запрос.
- Полное дообучение qwen2.5:3b — ~46 ГБ памяти. **LoRA** учит поправку
  `(alpha/r)·B·A` к замороженным весам: r=16 на все линейные слои — ~30 млн
  параметров (<1%), адаптер ~57 МБ.
- **QLoRA** = та же LoRA поверх базы в 4 битах: обучение 3B влезает в ~4 ГБ VRAM —
  бесплатный Colab справится.
- Три страха: забывание (LoRA мягче — адаптер снимается), переобучение (45 примеров!
  малый r, мало эпох, отложенный набор), дрейф базы (digest из урока 5.6 — рядом
  с адаптером).
- Метрика готова: `style_score` по пяти якорям + отложенные вопросы
  `test_questions.jsonl`.

**Дальше — урок 6.2:** превращаем 45 примеров в обучающий датасет (ChatML,
train/val, подсчёт токенов), пишем ноутбук для бесплатного Colab (unsloth + QLoRA)
и обучаем адаптер по-настоящему.

## 📚 Что почитать

- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) — первоисточник
- [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314) — NF4 и double quantization
- [Документация PEFT](https://huggingface.co/docs/peft) — LoraConfig и target_modules
- [Unsloth](https://docs.unsloth.ai/) — то, чем будем обучать в 6.2
